# opentelemetry_exporter_prometheus smoke test

Exercises the OpenTelemetry Prometheus exporter primary API.

In [ ]:
# Example 1: Public API surface — verify PrometheusMetricReader is exposed.
from opentelemetry.exporter.prometheus import PrometheusMetricReader

reader = PrometheusMetricReader()
print(f"reader = {reader!r}")
print(f"type(reader) = {type(reader).__name__}")

# PrometheusMetricReader is a MetricReader subclass.
from opentelemetry.sdk.metrics.export import MetricReader

assert isinstance(reader, MetricReader)
print("PrometheusMetricReader is a MetricReader subclass")

In [ ]:
# Example 2: Register the reader with a MeterProvider and create a counter.
from opentelemetry.exporter.prometheus import PrometheusMetricReader
from opentelemetry.sdk.metrics import MeterProvider

reader = PrometheusMetricReader()
provider = MeterProvider(metric_readers=[reader])
meter = provider.get_meter("smoketest.meter")

counter = meter.create_counter(
    name="smoketest_counter",
    unit="1",
    description="a counter for the smoke test",
)
counter.add(3, attributes={"env": "test"})
counter.add(7, attributes={"env": "test"})
print(f"counter = {counter!r}")
print("counter increments accepted")

In [ ]:
# Example 3: Verify metrics are exported into the prometheus_client registry.
from prometheus_client import REGISTRY

from opentelemetry.exporter.prometheus import PrometheusMetricReader
from opentelemetry.sdk.metrics import MeterProvider

reader = PrometheusMetricReader()
provider = MeterProvider(metric_readers=[reader])
meter = provider.get_meter("smoketest.export")
counter = meter.create_counter("export_test_counter", unit="1")
counter.add(42, attributes={"shard": "a"})

# Collecting from the global registry triggers the exporter callback.
names = {m.name for m in REGISTRY.collect()}
print(f"collected {len(names)} metric families from REGISTRY")
matching = [n for n in names if "export_test_counter" in n]
print(f"matching counter exports: {matching}")
assert matching, sorted(names)

In [ ]:
# Example 4: Histogram export via PrometheusMetricReader.
from prometheus_client import REGISTRY

from opentelemetry.exporter.prometheus import PrometheusMetricReader
from opentelemetry.sdk.metrics import MeterProvider

reader = PrometheusMetricReader()
provider = MeterProvider(metric_readers=[reader])
meter = provider.get_meter("smoketest.hist")
hist = meter.create_histogram("smoketest_latency", unit="ms")
for v in (1.0, 5.0, 12.5, 80.0, 200.0):
    hist.record(v, attributes={"route": "/x"})

names = {m.name for m in REGISTRY.collect()}
hist_matches = [n for n in names if "smoketest_latency" in n]
print(f"histogram metric names exported: {hist_matches}")
print(f"total metric families in registry: {len(names)}")
assert hist_matches, sorted(names)

In [ ]:
# Example 5: Render Prometheus text exposition format end-to-end.
from prometheus_client import REGISTRY, generate_latest

from opentelemetry.exporter.prometheus import PrometheusMetricReader
from opentelemetry.sdk.metrics import MeterProvider

reader = PrometheusMetricReader()
provider = MeterProvider(metric_readers=[reader])
meter = provider.get_meter("smoketest.render")
counter = meter.create_counter("render_test_counter", unit="1")
counter.add(11, attributes={"phase": "final"})

payload = generate_latest(REGISTRY).decode("utf-8")
print(f"exposition payload length = {len(payload)} chars")
print("--- preview ---")
preview_lines = [ln for ln in payload.splitlines() if "render_test_counter" in ln][:5]
for ln in preview_lines:
    print(ln)

assert "render_test_counter" in payload, "counter missing from exposition"
print("\nPrometheus exporter end-to-end OK")